# Pruebas de la logica de fix_giss_duplicado.sh

Prueba, SIN tocar nada real ni pegarle a ESGF, las partes del script
que manipulan archivos: filtrado del catalogo (paso 2), filtrado del
json de archivos (paso 2) y deteccion de realizaciones mezcladas
(verificacion post-descarga). Trabaja sobre COPIAS en el scratchpad,
no sobre data/interim/ real.

Los pasos 3 (busqueda ESGF) y 4/5 (descarga + reprocesamiento) son de
red/estado real -- no se simulan aca, se verifican con los propios
chequeos del script (aborta si algo no cierra).

In [1]:
import csv
import json
import re
import shutil
from pathlib import Path

SCRATCH = Path("/tmp/claude-1002/-home-jonathan-personal-e1-smn/c73a53d6-e4c9-42fa-b192-d6780205391c/scratchpad/fix_giss_test")
CATALOG_CSV = SCRATCH / "models_catalog_status.csv"
FILES_JSON = SCRATCH / "esgf_file_urls.json"
TARGET_MODELS = ["GISS-E2-1-G", "GISS-E2-1-H"]

assert CATALOG_CSV.exists() and FILES_JSON.exists(), "copias del scratchpad no encontradas"
print("trabajando sobre copias en", SCRATCH)

trabajando sobre copias en /tmp/claude-1002/-home-jonathan-personal-e1-smn/c73a53d6-e4c9-42fa-b192-d6780205391c/scratchpad/fix_giss_test


## Paso 2a: filtrado del catalogo (misma logica que el .sh)

In [2]:
def filtrar_catalogo(path: Path, targets: set[str]) -> tuple[int, int]:
    rows = list(csv.DictReader(open(path, newline="")))
    fields = list(rows[0].keys())
    before = len(rows)
    kept = [r for r in rows if r["model"] not in targets]
    w = csv.DictWriter(open(path, "w", newline=""), fieldnames=fields)
    w.writeheader()
    w.writerows(kept)
    return before, len(kept)

before, after = filtrar_catalogo(CATALOG_CSV, set(TARGET_MODELS))
print(f"catalogo: {before} -> {after} filas (esperado: -{len(TARGET_MODELS)})")
assert before - after == len(TARGET_MODELS), "no se removieron exactamente los 2 modelos esperados"

restantes = {r["model"] for r in csv.DictReader(open(CATALOG_CSV, newline=""))}
assert not (restantes & set(TARGET_MODELS)), "GISS-E2-1-G/H todavia aparecen en el catalogo filtrado"
print("OK: ningun TARGET_MODEL quedo en el catalogo filtrado")

catalogo: 41 -> 39 filas (esperado: -2)
OK: ningun TARGET_MODEL quedo en el catalogo filtrado


## Paso 2b: filtrado del json de archivos (misma logica que el .sh)

In [3]:
def filtrar_json(path: Path, targets: list[str]) -> tuple[int, int]:
    d = json.load(open(path))
    before = len(d)
    for m in targets:
        d.pop(m, None)
    json.dump(d, open(path, "w"))
    return before, len(d)

before, after = filtrar_json(FILES_JSON, TARGET_MODELS)
print(f"json: {before} -> {after} modelos (esperado: -{len(TARGET_MODELS)})")
assert before - after == len(TARGET_MODELS)

d = json.load(open(FILES_JSON))
assert not (set(d.keys()) & set(TARGET_MODELS)), "GISS-E2-1-G/H todavia aparecen en el json filtrado"
print("OK: ningun TARGET_MODEL quedo en el json filtrado")

json: 40 -> 38 modelos (esperado: -2)
OK: ningun TARGET_MODEL quedo en el json filtrado


## Verificacion post-descarga: contar realizaciones distintas por nombre de archivo

Mismo regex que el .sh (`r[0-9]+i[0-9]+p[0-9]+f[0-9]+`). Se prueba
contra: (a) nombres reales de un modelo local limpio (debe dar 1), y
(b) una lista sintetica con 2 realizaciones mezcladas, como la que
causo el bug en el HPC (debe dar >1, y disparar el aborto).

In [4]:
REALIZACION_RE = re.compile(r"(?<=_)r[0-9]+i[0-9]+p[0-9]+f[0-9]+(?=_)")

def contar_realizaciones(filenames: list[str]) -> int:
    members = {REALIZACION_RE.search(f).group() for f in filenames if REALIZACION_RE.search(f)}
    return len(members)

# (a) caso limpio, nombres reales de ACCESS-CM2 en local
import glob
limpios = [Path(f).name for f in glob.glob("/home/jonathan/personal/e1_smn/data/raw/cmip6/ACCESS-CM2/historical/*.nc")]
n_limpio = contar_realizaciones(limpios)
print(f"caso limpio (ACCESS-CM2 historical, {len(limpios)} archivos): {n_limpio} realizacion(es) -- {'OK' if n_limpio == 1 else 'FALLO'}")
assert n_limpio == 1

# (b) caso sintetico con el bug real reportado (r1i1p1f2 + r1i1p3f1)
sinteticos = [
    "tos_Omon_GISS-E2-1-G_historical_r1i1p1f2_gn_185001-190012.nc",
    "tos_Omon_GISS-E2-1-G_historical_r1i1p3f1_gn_185001-190012.nc",
]
n_mezclado = contar_realizaciones(sinteticos)
print(f"caso mezclado (sintetico, 2 archivos): {n_mezclado} realizacion(es) -- {'OK, dispara el aborto' if n_mezclado > 1 else 'FALLO, no lo detecta'}")
assert n_mezclado > 1

caso limpio (ACCESS-CM2 historical, 1 archivos): 1 realizacion(es) -- OK
caso mezclado (sintetico, 2 archivos): 2 realizacion(es) -- OK, dispara el aborto


## Extraccion de lista de archivos+urls del json (misma logica que el paso 4)

In [5]:
# Restaurar la copia del json (el paso anterior le saco GISS-E2-1-G/H,
# probamos la extraccion con un modelo que si sigue ahi)
d = json.load(open(FILES_JSON))
algun_modelo = next(iter(d))
algun_exp = next(iter(d[algun_modelo]))
entries = d[algun_modelo][algun_exp]
print(f"{algun_modelo} / {algun_exp}: {len(entries)} archivo(s) listado(s)")
print("primer archivo:", entries[0]["filename"] if entries else "(vacio)")
print("tiene 'urls' como lista:", isinstance(entries[0]["urls"], list) if entries else None)

ACCESS-CM2 / historical: 1 archivo(s) listado(s)
primer archivo: tos_Omon_ACCESS-CM2_historical_r1i1p1f1_gn_185001-201412.nc
tiene 'urls' como lista: True


## Limpieza

In [6]:
print("Copias de prueba en:", SCRATCH, "-- no afectan data/interim/ real. Borrar a mano si ya no hacen falta:")
print(f"  rm -rf {SCRATCH}")

Copias de prueba en: /tmp/claude-1002/-home-jonathan-personal-e1-smn/c73a53d6-e4c9-42fa-b192-d6780205391c/scratchpad/fix_giss_test -- no afectan data/interim/ real. Borrar a mano si ya no hacen falta:
  rm -rf /tmp/claude-1002/-home-jonathan-personal-e1-smn/c73a53d6-e4c9-42fa-b192-d6780205391c/scratchpad/fix_giss_test
